In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import to_date, lit, date_format

try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("BookInstallments") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/17 17:00:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
installments = spark.read \
    .parquet("/data/processed/installments")

installments.createOrReplaceTempView("installments")

# Contagem de linhas e colunas
num_rows = installments.count()
num_columns = len(installments.columns)

print(f'Quantidade de linhas: {num_rows}')
print(f'Quantidade de variaveis (colunas): {num_columns}')

installments.show(5, truncate=False)

Quantidade de linhas: 13605401
Quantidade de variaveis (colunas): 7
+-------------------+----------+------------------------------+--------------------------+-----------------------+-------------------+------------------+
|PK_AGG_INSTALLMENTS|SK_ID_CURR|FVL_DAYS_ENTRY_PAYMENT_INSTALM|FVL_AMT_INSTALMENT_INSTALM|FVL_AMT_PAYMENT_INSTALM|PK_DATREF_INSTALM  |PK_DATPART_INSTALM|
+-------------------+----------+------------------------------+--------------------------+-----------------------+-------------------+------------------+
|1135379            |366142    |-209.0                        |35758.485                 |35758.485              |2023-05-10 00:00:00|202305            |
|1849293            |308244    |-210.0                        |2021.445                  |2021.445               |2023-05-05 00:00:00|202305            |
|2262465            |388494    |-214.0                        |14052.375                 |14052.375              |2023-05-22 00:00:00|202305            |
|2482592

In [11]:
installments_01 = spark.sql("""
    SELECT
        PK_AGG_INSTALLMENTS,

        -- Estatísticas gerais de valores dos créditos anteriores
        min(FVL_DAYS_ENTRY_PAYMENT_INSTALM) as MIN_FVL_DAYS_ENTRY_PAYMENT_INSTALM,
        max(FVL_DAYS_ENTRY_PAYMENT_INSTALM) as MAX_FVL_DAYS_ENTRY_PAYMENT_INSTALM,
        avg(FVL_DAYS_ENTRY_PAYMENT_INSTALM) as AVG_FVL_DAYS_ENTRY_PAYMENT_INSTALM,


        min(FVL_AMT_INSTALMENT_INSTALM) as MIN_FVL_AMT_INSTALMENT_INSTALM,
        max(FVL_AMT_INSTALMENT_INSTALM) as MAX_FVL_AMT_INSTALMENT_INSTALM,
        avg(FVL_AMT_INSTALMENT_INSTALM) as AVG_FVL_AMT_INSTALMENT_INSTALM,


        min(FVL_AMT_PAYMENT_INSTALM) as MIN_FVL_AMT_PAYMENT_INSTALM,
        max(FVL_AMT_PAYMENT_INSTALM) as MAX_FVL_AMT_PAYMENT_INSTALM,
        avg(FVL_AMT_PAYMENT_INSTALM) as AVG_FVL_AMT_PAYMENT_INSTALM,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 3 meses (out/nov/dez 2023)
        avg(case when PK_DATPART_INSTALM in (202310, 202311, 202312)
                 then FVL_DAYS_ENTRY_PAYMENT_INSTALM else null end) as AVG_DAYS_ENTRY_PAYMENT_U3M_INSTALM,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 6 meses (jul a dez 2023)
        avg(case when PK_DATPART_INSTALM in (202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_DAYS_ENTRY_PAYMENT_INSTALM else null end) as AVG_DAYS_ENTRY_PAYMENT_U6M_INSTALM,

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 9 meses (abr a dez 2023)
        avg(case when PK_DATPART_INSTALM in (202304, 202305, 202306, 202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_DAYS_ENTRY_PAYMENT_INSTALM else null end) as AVG_DAYS_ENTRY_PAYMENT_U9M_INSTALM, 

        -- Média do tempo de pagamento de créditos anteriores por janelas de tempo nos últimos 12 meses (jan a dez 2023)
        avg(case when PK_DATPART_INSTALM in (202301, 202302, 202303, 202304, 202305, 202306,
                                                                      202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_DAYS_ENTRY_PAYMENT_INSTALM else null end) as AVG_DAYS_ENTRY_PAYMENT_U12M_INSTALM,    


        -- Média dos valores de parcelas previstas de créditos anteriores por janelas de tempo nos últimos 3 meses (out/nov/dez 2023)
        avg(case when PK_DATPART_INSTALM in (202310, 202311, 202312)
                 then FVL_AMT_INSTALMENT_INSTALM else null end) as AVG_AMT_INSTALMENT_U3M_INSTALM,

        -- Média dos valores de parcelas previstas de créditos anteriores por janelas de tempo nos últimos 6 meses (jul a dez 2023)
        avg(case when PK_DATPART_INSTALM in (202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_INSTALMENT_INSTALM else null end) as AVG_AMT_INSTALMENT_U6M_INSTALM,

        -- Média dos valores de parcelas previstas de créditos anteriores por janelas de tempo nos últimos 9 meses (abr a dez 2023)
        avg(case when PK_DATPART_INSTALM in (202304, 202305, 202306, 202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_INSTALMENT_INSTALM else null end) as AVG_AMT_INSTALMENT_U9M_INSTALM, 

        -- Média dos valores de parcelas previstas de créditos anteriores por janelas de tempo nos últimos 12 meses (jan a dez 2023)
        avg(case when PK_DATPART_INSTALM in (202301, 202302, 202303, 202304, 202305, 202306,
                                                                      202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_INSTALMENT_INSTALM else null end) as AVG_AMT_INSTALMENT_U12M_INSTALM,       



        -- Média dos valores efetivamente pagos em créditos anteriores por janelas de tempo nos últimos 3 meses (out/nov/dez 2023)
        avg(case when PK_DATPART_INSTALM in (202310, 202311, 202312)
                 then FVL_AMT_PAYMENT_INSTALM else null end) as AVG_AMT_PAYMENT_U3M_INSTALM,

        -- Média dos valores efetivamente pagos em créditos anteriores por janelas de tempo nos últimos 6 meses (jul a dez 2023)
        avg(case when PK_DATPART_INSTALM in (202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_PAYMENT_INSTALM else null end) as AVG_AMT_PAYMENT_U6M_INSTALM,

        -- Média dos valores efetivamente pagos em créditos anteriores por janelas de tempo nos últimos 9 meses (abr a dez 2023)
        avg(case when PK_DATPART_INSTALM in (202304, 202305, 202306, 202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_PAYMENT_INSTALM else null end) as AVG_AMT_PAYMENT_U9M_INSTALM, 

        -- Média dos valores efetivamente pagos em créditos anteriores por janelas de tempo nos últimos 12 meses (jan a dez 2023)
        avg(case when PK_DATPART_INSTALM in (202301, 202302, 202303, 202304, 202305, 202306,
                                                                      202307, 202308, 202309, 202310, 202311, 202312)
                 then FVL_AMT_PAYMENT_INSTALM else null end) as AVG_AMT_PAYMENT_U12M_INSTALM                

    FROM installments
    GROUP BY PK_AGG_INSTALLMENTS
""")

installments_01.createOrReplaceTempView("installments_01")
num_rows = installments_01.count()
num_columns = len(installments_01.columns)

print(f'Quantidade de linhas: {num_rows}')
print(f'Quantidade de variaveis (colunas): {num_columns}')

[Stage 24:==>                                                      (1 + 6) / 20]

Quantidade de linhas: 997752
Quantidade de variaveis (colunas): 22


In [8]:
installments_01.show(5, truncate=False)

25/07/17 17:22:10 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------------------+----------------------------------+----------------------------------+----------------------------------+------------------------------+------------------------------+------------------------------+---------------------------+---------------------------+---------------------------+----------------------------------+----------------------------------+----------------------------------+-----------------------------------+------------------------------+------------------------------+------------------------------+-------------------------------+---------------------------+---------------------------+---------------------------+----------------------------+
|PK_AGG_INSTALLMENTS|MIN_FVL_DAYS_ENTRY_PAYMENT_INSTALM|MAX_FVL_DAYS_ENTRY_PAYMENT_INSTALM|AVG_FVL_DAYS_ENTRY_PAYMENT_INSTALM|MIN_FVL_AMT_INSTALMENT_INSTALM|MAX_FVL_AMT_INSTALMENT_INSTALM|AVG_FVL_AMT_INSTALMENT_INSTALM|MIN_FVL_AMT_PAYMENT_INSTALM|MAX_FVL_AMT_PAYMENT_INSTALM|AVG_FVL_AMT_PAYMENT_INSTALM|AVG_DAYS_ENTRY

In [12]:
installments_02 = spark.sql("""
    SELECT
        *,
        -- Razões entre médias por janelas de tempo (tendência temporal)
        round(AVG_DAYS_ENTRY_PAYMENT_U3M_INSTALM/AVG_DAYS_ENTRY_PAYMENT_U6M_INSTALM,2) as VL_RAZ_MED_U3M_U6M_DAYS_ENTRY_PAYMENT_INSTALM,
        round(AVG_DAYS_ENTRY_PAYMENT_U6M_INSTALM/AVG_DAYS_ENTRY_PAYMENT_U9M_INSTALM,2) as VL_RAZ_MED_U6M_U9M_DAYS_ENTRY_PAYMENT_INSTALM,
        round(AVG_DAYS_ENTRY_PAYMENT_U9M_INSTALM/AVG_DAYS_ENTRY_PAYMENT_U12M_INSTALM,2) as VL_RAZ_MED_U9M_U12M_DAYS_ENTRY_PAYMENT_INSTALM,

        round(AVG_AMT_INSTALMENT_U3M_INSTALM/AVG_AMT_INSTALMENT_U6M_INSTALM,2) as VL_RAZ_MED_U3M_U6M_AMT_INSTALMENT_INSTALM,
        round(AVG_AMT_INSTALMENT_U6M_INSTALM/AVG_AMT_INSTALMENT_U9M_INSTALM,2) as VL_RAZ_MED_U6M_U9M_AMT_INSTALMENT_INSTALM,
        round(AVG_AMT_INSTALMENT_U9M_INSTALM/AVG_AMT_INSTALMENT_U12M_INSTALM,2) as VL_RAZ_MED_U9M_U12M_AMT_INSTALMENT_INSTALM,

        round(AVG_AMT_PAYMENT_U3M_INSTALM/AVG_AMT_PAYMENT_U6M_INSTALM,2) as VL_RAZ_MED_U3M_U6M_AMT_PAYMENT_INSTALM,
        round(AVG_AMT_PAYMENT_U6M_INSTALM/AVG_AMT_PAYMENT_U9M_INSTALM,2) as VL_RAZ_MED_U6M_U9M_AMT_PAYMENT_INSTALM,
        round(AVG_AMT_PAYMENT_U9M_INSTALM/AVG_AMT_PAYMENT_U12M_INSTALM,2) as VL_RAZ_MED_U9M_U12M_AMT_PAYMENT_INSTALM,

        -- Razão entre valores pagos e parcelas previstas (indicador de adimplência)
        round(AVG_AMT_PAYMENT_U3M_INSTALM/AVG_AMT_INSTALMENT_U3M_INSTALM,2) as VL_RAZ_MED_U3M_PAYMENT_TO_INSTALMENT_INSTALM,
        round(AVG_AMT_PAYMENT_U6M_INSTALM/AVG_AMT_INSTALMENT_U6M_INSTALM,2) as VL_RAZ_MED_U6M_PAYMENT_TO_INSTALMENT_INSTALM,
        round(AVG_AMT_PAYMENT_U9M_INSTALM/AVG_AMT_INSTALMENT_U9M_INSTALM,2) as VL_RAZ_MED_U9M_PAYMENT_TO_INSTALMENT_INSTALM, 
        round(AVG_AMT_PAYMENT_U12M_INSTALM/AVG_AMT_INSTALMENT_U12M_INSTALM,2) as VL_RAZ_MED_U12M_PAYMENT_TO_INSTALMENT_INSTALM 

    FROM installments_01

""")

installments_02.createOrReplaceTempView("installments_02")
num_rows = installments_02.count()
num_columns = len(installments_02.columns)

print(f'Quantidade de linhas: {num_rows}')
print(f'Quantidade de variaveis (colunas): {num_columns}')

Quantidade de linhas: 997752
Quantidade de variaveis (colunas): 35


In [10]:
installments_02.show(5, truncate=False)

+-------------------+----------------------------------+----------------------------------+----------------------------------+------------------------------+------------------------------+------------------------------+---------------------------+---------------------------+---------------------------+----------------------------------+----------------------------------+----------------------------------+-----------------------------------+------------------------------+------------------------------+------------------------------+-------------------------------+---------------------------+---------------------------+---------------------------+----------------------------+---------------------------------------------+---------------------------------------------+----------------------------------------------+-----------------------------------------+-----------------------------------------+------------------------------------------+--------------------------------------+-------------

# 📘 Dicionário de Variáveis - Book Installments

| Variável                                         | Descrição                                                                               |
| ------------------------------------------------ | --------------------------------------------------------------------------------------- |
| `MIN_FVL_DAYS_ENTRY_PAYMENT_INSTALM`             | Menor tempo entre o pagamento da parcela e a data de solicitação do empréstimo atual    |
| `MAX_FVL_DAYS_ENTRY_PAYMENT_INSTALM`             | Maior tempo entre o pagamento da parcela e a data de solicitação do empréstimo atual    |
| `AVG_FVL_DAYS_ENTRY_PAYMENT_INSTALM`             | Tempo médio entre o pagamento da parcela e a data de solicitação do empréstimo atual    |
| `MIN_FVL_AMT_INSTALMENT_INSTALM`                 | Menor valor de parcela prevista em créditos anteriores                                  |
| `MAX_FVL_AMT_INSTALMENT_INSTALM`                 | Maior valor de parcela prevista em créditos anteriores                                  |
| `AVG_FVL_AMT_INSTALMENT_INSTALM`                 | Valor médio das parcelas previstas em créditos anteriores                               |
| `MIN_FVL_AMT_PAYMENT_INSTALM`                    | Menor valor efetivamente pago em créditos anteriores                                    |
| `MAX_FVL_AMT_PAYMENT_INSTALM`                    | Maior valor efetivamente pago em créditos anteriores                                    |
| `AVG_FVL_AMT_PAYMENT_INSTALM`                    | Valor médio efetivamente pago em créditos anteriores                                    |
| `AVG_DAYS_ENTRY_PAYMENT_U3M_INSTALM`             | Tempo médio entre o pagamento e a solicitação do crédito atual nos últimos 3 meses      |
| `AVG_DAYS_ENTRY_PAYMENT_U6M_INSTALM`             | Tempo médio entre o pagamento e a solicitação do crédito atual nos últimos 6 meses      |
| `AVG_DAYS_ENTRY_PAYMENT_U9M_INSTALM`             | Tempo médio entre o pagamento e a solicitação do crédito atual nos últimos 9 meses      |
| `AVG_DAYS_ENTRY_PAYMENT_U12M_INSTALM`            | Tempo médio entre o pagamento e a solicitação do crédito atual nos últimos 12 meses     |
| `AVG_AMT_INSTALMENT_U3M_INSTALM`                 | Valor médio das parcelas previstas nos últimos 3 meses                                  |
| `AVG_AMT_INSTALMENT_U6M_INSTALM`                 | Valor médio das parcelas previstas nos últimos 6 meses                                  |
| `AVG_AMT_INSTALMENT_U9M_INSTALM`                 | Valor médio das parcelas previstas nos últimos 9 meses                                  |
| `AVG_AMT_INSTALMENT_U12M_INSTALM`                | Valor médio das parcelas previstas nos últimos 12 meses                                 |
| `AVG_AMT_PAYMENT_U3M_INSTALM`                    | Valor médio efetivamente pago nos últimos 3 meses                                       |
| `AVG_AMT_PAYMENT_U6M_INSTALM`                    | Valor médio efetivamente pago nos últimos 6 meses                                       |
| `AVG_AMT_PAYMENT_U9M_INSTALM`                    | Valor médio efetivamente pago nos últimos 9 meses                                       |
| `AVG_AMT_PAYMENT_U12M_INSTALM`                   | Valor médio efetivamente pago nos últimos 12 meses                                      |
| `VL_RAZ_MED_U3M_U6M_DAYS_ENTRY_PAYMENT_INSTALM`  | Razão entre médias de tempo de pagamento: 3 meses vs. 6 meses                           |
| `VL_RAZ_MED_U6M_U9M_DAYS_ENTRY_PAYMENT_INSTALM`  | Razão entre médias de tempo de pagamento: 6 meses vs. 9 meses                           |
| `VL_RAZ_MED_U9M_U12M_DAYS_ENTRY_PAYMENT_INSTALM` | Razão entre médias de tempo de pagamento: 9 meses vs. 12 meses                          |
| `VL_RAZ_MED_U3M_U6M_AMT_INSTALMENT_INSTALM`      | Razão entre médias de parcelas previstas: 3 meses vs. 6 meses                           |
| `VL_RAZ_MED_U6M_U9M_AMT_INSTALMENT_INSTALM`      | Razão entre médias de parcelas previstas: 6 meses vs. 9 meses                           |
| `VL_RAZ_MED_U9M_U12M_AMT_INSTALMENT_INSTALM`     | Razão entre médias de parcelas previstas: 9 meses vs. 12 meses                          |
| `VL_RAZ_MED_U3M_U6M_AMT_PAYMENT_INSTALM`         | Razão entre médias de valores pagos: 3 meses vs. 6 meses                                |
| `VL_RAZ_MED_U6M_U9M_AMT_PAYMENT_INSTALM`         | Razão entre médias de valores pagos: 6 meses vs. 9 meses                                |
| `VL_RAZ_MED_U9M_U12M_AMT_PAYMENT_INSTALM`        | Razão entre médias de valores pagos: 9 meses vs. 12 meses                               |
| `VL_RAZ_MED_U3M_PAYMENT_TO_INSTALMENT_INSTALM`   | Razão entre valor pago e valor previsto nos últimos 3 meses (indicador de adimplência)  |
| `VL_RAZ_MED_U6M_PAYMENT_TO_INSTALMENT_INSTALM`   | Razão entre valor pago e valor previsto nos últimos 6 meses (indicador de adimplência)  |
| `VL_RAZ_MED_U9M_PAYMENT_TO_INSTALMENT_INSTALM`   | Razão entre valor pago e valor previsto nos últimos 9 meses (indicador de adimplência)  |
| `VL_RAZ_MED_U12M_PAYMENT_TO_INSTALMENT_INSTALM`  | Razão entre valor pago e valor previsto nos últimos 12 meses (indicador de adimplência) |


#### Salvando tabela particionada (Parquet)

In [14]:
nm_path = '/data/books/installments'
installments_02.write.parquet(nm_path, mode='overwrite')
#bureau_etl_01.coalesce(1).write.parquet(nm_path, mode="overwrite")

In [15]:
spark.stop()